In [20]:
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
#Define the project root and raw data directory
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [8]:
#Read the orders CSV file into a DataFrame
df_orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv", dtype=str)

In [9]:
df_orders.shape

(99441, 8)

In [10]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [30]:
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [43]:
df_orders['order_id'].is_unique
df_orders['order_id'].isnull().sum()

np.int64(0)

### **Orders Dataset - Initial Data Profiling**

**Dataset Overview**

The orders dataset contains 99.441 rows and 8 columns. It stores order identifiers, order status, and timestamps associated with the order lifecycle.

**Grain and Key Validation**

The dataset grain is one row per order.

The `order_id` column is a validated primary key candidate for the current dataset:

- *Uniqueness:* all 99,441 values are distinct.
- *Completeness:* the column contains no missing values.

These checks support the expected grain of one row per order.

**Missing Values**

Missing values were identified in the following timestamp columns:

- `order_approved_at`
- `order_delivered_carrier_date`
- `order_delivered_customer_date`

These missing values require further investigation against `order_status` to distinguish expected lifecycle states from potential data quality issues.

No missing values have been filled, and no rows have been removed at this stage.



In [56]:
na_order_approved_at = df_orders['order_approved_at'].isna().groupby(df_orders['order_status']).sum()
na_order_delivered_carrier_date = df_orders['order_delivered_carrier_date'].isna().groupby(df_orders['order_status']).sum()
na_order_delivered_customer_date = df_orders['order_delivered_customer_date'].isna().groupby(df_orders['order_status']).sum()
orders_by_status = df_orders['order_status'].value_counts()

In [ ]:
#Investigating the number of missing values by order status for the columns 'order_approved_at', 'order_delivered_carrier_date', and 'order_delivered_customer_date'
na_comparision_by_status = pd.concat([orders_by_status.rename("total_orders"),
                                     na_order_approved_at.rename("na_order_approved_at"),
                                      na_order_delivered_carrier_date.rename("na_order_delivered_carrier_date"),
                                      na_order_delivered_customer_date.rename("na_order_delivered_customer_date")
                                    ],  
                                      axis=1)

na_comparision_by_status

,total_orders,na_order_approved_at,na_order_delivered_carrier_date,na_order_delivered_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2
